<a href="https://colab.research.google.com/github/lidia-notebook/nlp-ner-bahasa-indonesia/blob/main/NLPLidia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini Project: NLP - Named Entity Recognition (NER)


---

**Name:** Lidia Priskila
**Period:** Generative AI · Computer Vision · **Natural Language Processing (NLP)**  
**Tools:** Python, HuggingFace Transformers, Google Colab

## 1. Project Description

This project aims to build a **Named Entity Recognition (NER)** model for Indonesian text
using **fine-tuning** on a pre-trained multilingual BERT model.

The model will be trained to detect and classify the following entities:

| Label | Description | Example |
|-------|-------------|---------|
| `PER` | Person name | Jokowi, Prabowo Subianto |
| `LOC` | Location / place | Jakarta, Bandung, Jawa Barat |
| `ORG` | Organization / institution | TNI, Bank Indonesia, Kompas |
| `O`   | Non-entity | conjunctions, prepositions, etc. |

## 2. Dataset

**Source:** WikiANN (Indonesian) via HuggingFace Datasets  
**Link:** https://huggingface.co/datasets/unimelb-nlp/wikiann  

This is a public NER dataset based on Wikipedia, annotated using **BIO tagging** format:

- `B-PER`, `I-PER` → person entity
- `B-LOC`, `I-LOC` → location entity
- `B-ORG`, `I-ORG` → organization entity
- `O` → non-entity

**Why WikiANN?**
- No manual download needed — load directly via code
- Already in BIO format, ready for fine-tuning
- Available splits: train, validation, test

## 3. Model

**Model:** `bert-base-multilingual-cased`  
**Source:** https://huggingface.co/google-bert/bert-base-multilingual-cased

BERT (Bidirectional Encoder Representations from Transformers) is a language model
by Google, pre-trained on 104 languages including Indonesian.

**Fine-tuning pipeline:**

```
[WikiANN Dataset (id)]
        ↓
[Tokenization & BIO Label Alignment]
        ↓
[Fine-tune bert-base-multilingual-cased]
        ↓
[Evaluation: Precision / Recall / F1]
        ↓
[Error Analysis & Insights]
```

## 4. Objectives

1. Select and load a quality Indonesian NER dataset
2. Perform preprocessing and BIO tagging alignment for BERT
3. Visualize data distribution and entity statistics
4. Fine-tune `bert-base-multilingual-cased` for NER task
5. Evaluate the model using Precision, Recall, and F1-Score per entity
6. Analyze prediction results: correct, incorrect, and error patterns
7. Provide insights and actionable recommendations based on model results

## 5. AI Ethics Considerations

### Data Security
The WikiANN dataset is sourced from Wikipedia, which is publicly available and free to use.
The dataset does not contain sensitive personal information such as ID numbers,
home addresses, or phone numbers.

### User Privacy
Although the dataset is from public sources, this project ensures that individual
entities are not used in any privacy-violating manner. No personal data is collected.

### AI Ethics & Bias
BERT-based models may exhibit bias, including:
- Western names may be recognized more accurately than local names (e.g. Javanese or Papuan names)
- Model performance may vary across topics (politics, sports, culture)

Evaluation will be conducted across diverse topics to identify and document potential bias.

## 6. Project Workflow

| Phase | Content | Target |
|-------|---------|--------|
| **Phase 0** | Project Background | Understand context & objectives |
| **Phase 1** | Setup & Load Data | Environment ready, dataset loaded |
| **Phase 2** | Preprocessing | Tokenized data & aligned BIO labels |
| **Phase 3** | Fine-tuning | Trained model saved to disk |
| **Phase 4** | Evaluation | Precision, Recall, F1 per entity |
| **Phase 5** | Analysis & Insights | Error patterns, recommendations, AI ethics |

---


# Phase 1: Setup & Load Data

In [1]:
# install libraries
!pip install transformers datasets seqeval -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 630.7 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 179.2 kB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [3]:
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer
)

In [4]:
# import all libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import AutoTokenizer

then, we load WikiANN indonesian dataset

In [5]:
dataset = load_dataset("unimelb-nlp/wikiann", "id")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

id/validation-00000-of-00001.parquet:   0%|          | 0.00/550k [00:00<?, ?B/s]

id/test-00000-of-00001.parquet:   0%|          | 0.00/548k [00:00<?, ?B/s]

id/train-00000-of-00001.parquet:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})


In [6]:
from datasets import load_dataset

In [7]:
dataset = load_dataset("unimelb-nlp/wikiann", "id")
label_list = dataset['train'].features['ner_tags'].feature.names
print("Dataset loaded!")
print(dataset)

Dataset loaded!
DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})


After that, we quick explore the dataset structure

In [8]:
# Check dataset splits and one sample
print("=== Dataset Splits ===")
print(f"Train : {len(dataset['train'])} samples")
print(f"Validation: {len(dataset['validation'])} samples")
print(f"Test  : {len(dataset['test'])} samples")

print("\n=== Sample Data (train[0]) ===")
print(dataset['train'][0])

=== Dataset Splits ===
Train : 20000 samples
Validation: 10000 samples
Test  : 10000 samples

=== Sample Data (train[0]) ===
{'tokens': ['ALIH', 'Al-Hilal', '(', 'Omdurman', ')'], 'ner_tags': [0, 3, 4, 4, 4], 'langs': ['id', 'id', 'id', 'id', 'id'], 'spans': ['ORG: Al-Hilal ( Omdurman )']}


Now we could see the ..... THEN WE go to check NER label list

In [9]:
label_list = dataset['train'].features['ner_tags'].feature.names
print("NER Labels:", label_list)

NER Labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


# Phase 2: Preprocessing & BIO Label Alignment

Its started by loading the tokenizer from pretrained multilingual BERT

In [10]:
model_checkpoint = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
print(f"Tokenizer loaded: {model_checkpoint}")
print(f"Vocab size: {tokenizer.vocab_size}")

Tokenizer loaded: bert-base-multilingual-cased
Vocab size: 119547


We do need label alignment **because,**

BERT splits words into **subwords** (wordpieces). For example:
- "Subianto" → ["Su", "##bian", "##to"] (3 subword tokens)

But the original label is just **1 label** for the whole word.
So we need to **align** the labels to match the subword tokens:
- "Su"      → B-PER
- "##bian"  → I-PER  (or -100 to ignore)
- "##to"    → I-PER  (or -100 to ignore)

We use `-100` for subword continuations so the loss function ignores them during training. That's why next we are going to tokenize and align function

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_id = None
        label_ids = []

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100) # to ignore special tokens (CLS & SEP)
            elif word_id != previous_word_id:
                label_ids.append(labels[word_id])
            else:
                label_ids.append(-100)
            previous_word_id = word_id

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

Then we apply tokenization to train, validation, and test splits

In [13]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True
)


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [14]:

print("Tokenization complete!")
print(tokenized_dataset)

Tokenization complete!
DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 20000
    })
})


From the cells above we could see the dataset has been tokenized, and we continue to verify alignment

In [15]:
# try on one sample first
sample_idx = 0
tokens      = dataset['train'][sample_idx]['tokens']
ner_tags    = dataset['train'][sample_idx]['ner_tags']
label_list  = dataset['train'].features['ner_tags'].feature.names

In [16]:
# continue to tokenize at single sample
encoded = tokenizer(tokens, is_split_into_words=True)
word_ids = encoded.word_ids()
subwords = tokenizer.convert_ids_to_tokens(encoded['input_ids'])

In [17]:
print(f"{'Subword':<15} {'Word ID':<10} {'Label'}")
print("-" * 40)
for subword, word_id in zip(subwords, word_ids):
    if word_id is None:
        label = "[special]"
    else:
        label = label_list[ner_tags[word_id]]
    print(f"{subword:<15} {str(word_id):<10} {label}")

Subword         Word ID    Label
----------------------------------------
[CLS]           None       [special]
AL              0          O
##IH            0          O
Al              1          B-ORG
-               1          B-ORG
Hi              1          B-ORG
##lal           1          B-ORG
(               2          I-ORG
Om              3          I-ORG
##dur           3          I-ORG
##man           3          I-ORG
)               4          I-ORG
[SEP]           None       [special]


In simple terms:

- `[CLS]` and `[SEP]` → special BERT tokens, labeled as `[special]`, so they are ignored during training.
- `AL` + `##IH` → one word split into 2 subwords, both labeled `O` because they are not entities.
- `Al`, `-`, `Hi`, `##lal` → one organization entity, all labeled `B-ORG` because they share the same `word_id (=1)`.
- `(`, `Om##dur##man`, `)` → continuation of the same ORG entity, labeled `I-ORG`.

Main point: the alignment worked successfully. Each subword already receives the correct label based on its original word.


# Phase 3: Fine-Tuning the Model

First we are going to load pre-trained BERT model for token classification

In [18]:
from transformers import AutoModelForTokenClassification

In [19]:
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label={i: label for i, label in enumerate(label_list)},
    label2id={label: i for i, label in enumerate(label_list)}
)

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params

In [20]:
print(f"Model loaded: {model_checkpoint}")
print(f"Number of labels: {len(label_list)}")
print(f"Labels: {label_list}")

Model loaded: bert-base-multilingual-cased
Number of labels: 7
Labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


From the result, we can see

- **Number of labels: 7**
  → This matches your NER task:
  ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']

And regarding the WARNING (Important but Normal)
- **UNEXPECTED**
  → These are layers from pretraining (like MLM / next sentence prediction)
  → Not needed for token classification → SAFE to ignore

- **MISSING**
  → `classifier.weight` and `classifier.bias`
  → This is EXPECTED because:
    - Your task (NER) is different from original BERT task
    - So a new classification head is created from scratch

#### **In short,**

- Base BERT weights loaded correctly
- Task-specific classifier initialized (will be trained later)
- Model is READY for training

Which in the next step we are going define training arguments

In [21]:
from transformers import TrainingArguments

In [22]:
training_args = TrainingArguments(
    output_dir="./ner-bert-indonesian",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=50,
    report_to="none"
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [23]:
print("Training arguments defined!")
print(f"Epochs        : {training_args.num_train_epochs}")
print(f"Learning rate : {training_args.learning_rate}")
print(f"Batch size    : {training_args.per_device_train_batch_size}")

Training arguments defined!
Epochs        : 3
Learning rate : 2e-05
Batch size    : 16


**Explenations**

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `num_train_epochs` | 3 | Model will see the entire dataset 3 times |
| `learning_rate` | 2e-05 | How fast the model updates its weights — small = stable |
| `per_device_train_batch_size` | 16 | 16 samples processed per training step |
| `eval_strategy` | epoch | Evaluate model performance after every epoch |
| `load_best_model_at_end` | True | Automatically keep the best model based on F1 score |

Next, we are going define evaluation metrics using seqeval because Standard metrics like `accuracy` are **not suitable** for NER because:
- Most tokens are labeled `O` (non-entity), so accuracy is naturally high even if the model is bad
- Example: if 90% of tokens are `O`, a model that always predicts `O` gets 90% accuracy — but detects zero entities!

`seqeval` solves this by evaluating at the **entity level**, not token level:

| Metric | What it measures |
|--------|-----------------|
| **Precision** | Of all entities the model predicted, how many were correct? |
| **Recall** | Of all real entities in the text, how many did the model find? |
| **F1-Score** | Harmonic mean of Precision & Recall — the main metric for NER |

Example:
- Text has 10 entities
- Model predicts 8 → 7 correct
- Precision = 7/8 = 87.5%
- Recall = 7/10 = 70%
- F1 = 77.8% ← this is what we optimize

In [24]:
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report


In [25]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    true_labels = [
        [label_list[l] for l in label_row if l != -100]
        for label_row in labels
    ]
    true_predictions = [
        [label_list[p] for p, l in zip(pred_row, label_row) if l != -100]
        for pred_row, label_row in zip(predictions, labels)
    ]

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall"   : recall_score(true_labels, true_predictions),
        "f1"       : f1_score(true_labels, true_predictions),
    }

print("Metrics function defined!")

Metrics function defined!


Then we handle data collator first before training the model.



In [26]:
from transformers import DataCollatorForTokenClassification, Trainer

In training the model we initialize trainer and start fine-tuning it

In [27]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [28]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

Then start training

In [ ]:
print("Starting training...")
trainer.train()

Starting training...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss


# Phase 4: Model Evaluation

After fine-tuning, we need to measure how well the model actually performs on **unseen data** (test set).
Training loss going down does NOT always mean the model is good at the real task

We evaluate using 3 metrics per entity type:
- **Precision** → when the model says "this is an entity", how often is it right?
- **Recall** → of all real entities in the text, how many did the model find?
- **F1-Score** → the balance between Precision and Recall — our main metric

We evaluate on the **test set** (data the model has never seen during training)

In [ ]:
from seqeval.metrics import classification_report

In [ ]:
from seqeval.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
print("Running evaluation on test set...")
results = trainer.evaluate(eval_dataset=tokenized_dataset["test"])


In [ ]:
print("\n=== Evaluation Results ===")
for key, value in results.items():
    print(f"{key:<30} : {value:.4f}" if isinstance(value, float) else f"{key:<30} : {value}")

Overall F1 alone is not enough,thats why we need per-entity report. A model could be:
- Very good at detecting `LOC` (location) → F1 = 0.95
- But terrible at detecting `ORG` (organization) → F1 = 0.40

If we only look at the overall score, we miss this problem.
Per-entity breakdown helps us understand **where the model struggles**.

In [ ]:
# get predictions on test set
predictions, labels, _ = trainer.predict(tokenized_dataset["test"])
predictions = np.argmax(predictions, axis=-1)

In [ ]:
# convert to label strings with ignoring the -100 padding
true_labels = [
    [label_list[l] for l in label_row if l != -100]
    for label_row in labels
]
true_predictions = [
    [label_list[p] for p, l in zip(pred_row, label_row) if l != -100]
    for pred_row, label_row in zip(predictions, labels)
]

In [ ]:
print("=== Detailed Classification Report ===\n")
print(classification_report(true_labels, true_predictions))

Next we are going to visualizing the result so we could see better

In [ ]:
entities = ["PER", "LOC", "ORG"]
precisions, recalls, f1s = [], [], []

In [ ]:
for entity in entities:
    # Filter only labels belonging to this entity
    filtered_true = [[t for t in seq if entity in t or t == "O"] for seq in true_labels]
    filtered_pred = [[p for p, t in zip(pseq, tseq) if entity in t or t == "O"]
                     for pseq, tseq in zip(true_predictions, true_labels)]

    p = precision_score(true_labels, true_predictions, average="macro")
    r = recall_score(true_labels, true_predictions, average="macro")
    f = f1_score(true_labels, true_predictions, average="macro")
    precisions.append(round(p, 4))
    recalls.append(round(r, 4))
    f1s.append(round(f, 4))

x = range(len(entities))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar([i - width for i in x], precisions, width, color='steelblue', label='Precision')
bars2 = ax.bar([i for i in x],          recalls,    width, color='seagreen',  label='Recall')
bars3 = ax.bar([i + width for i in x],  f1s,        width, color='coral',     label='F1-Score')

ax.set_title('Precision, Recall & F1-Score per Entity Type', fontsize=14)
ax.set_xticks(list(x))
ax.set_xticklabels(entities)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.legend()

for bar in list(bars1) + list(bars2) + list(bars3):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

From the result above shown ......... and to close that, we are going to save predictions to dataframe. Since it will allow us to:
- manually inspect which sentences the model got wrong or right
- find patterns in mistakes (i.e., does the model confuse `ORG` with `LOC`?)
- use for later in Phase 5 for deeper error analyssis

In [ ]:
results_df = pd.DataFrame({
    "true_labels"      : [" ".join(seq) for seq in true_labels],
    "predicted_labels" : [" ".join(seq) for seq in true_predictions],
    "match"            : [seq_t == seq_p for seq_t, seq_p in zip(true_labels, true_predictions)]
})

In [ ]:
print(f"Total test samples  : {len(results_df)}")
print(f"Perfectly matched   : {results_df['match'].sum()}")
print(f"Accuracy (sentence) : {results_df['match'].mean():.2%}")
print()
results_df.head(10)

...

# Phase 5: Error Analysis & Insights

This phase is important since a good F1 score does not tell the whole story. Error analysis helps us understand:
- **What** the model gets wrong
- **Why** it gets it wrong
- **Where** to imporove in future iterations.

This help us to know what we could do next, what step that we will take.

In [ ]:
mismatched = results_df[results_df["match"] == False].reset_index(drop=True)


In [ ]:
print(f"Total mismatched sentences : {len(mismatched)}")
print(f"Total test sentences       : {len(results_df)}")
print(f"Error rate                 : {len(mismatched)/len(results_df):.2%}")
print()
mismatched.head(10)

The result shows..... then we continue to **Token Level Error Analysis**, which sentence-level mismatch is too broad. One wrong token in a 20-token sentence counts as a full mismatch. with token-level analysis it will helps us:
- exactly **which token** was predicted wrong
- what the model predicted vs what the correct label was
- whether errors cluster around specific entity types.

In [ ]:
error_records = []

for true_seq, pred_seq in zip(true_labels, true_predictions):
    for true_tag, pred_tag in zip(true_seq, pred_seq):
        if true_tag != pred_tag:
            error_records.append({
                "true_label" : true_tag,
                "pred_label" : pred_tag,
            })

error_df = pd.DataFrame(error_records)


In [ ]:
print(f"Total token-level errors: {len(error_df)}")
print()
print("=== Most Common Errors ===")
print(error_df.groupby(["true_label", "pred_label"])
      .size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
      .head(15)
      .to_string(index=False))

The result ...... and we make heatmap to help see the patterns more clearly.

In [ ]:
from sklearn.metrics import confusion_matrix

In [ ]:
# flatten all labels
all_true = [t for seq in true_labels for t in seq]
all_pred = [p for seq in true_predictions for p in seq]

In [ ]:
# get unique labels
unique_labels = sorted(set(all_true + all_pred))

In [ ]:
cm = confusion_matrix(all_true, all_pred, labels=unique_labels)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=unique_labels,
    yticklabels=unique_labels,
    cmap="Blues",
    linewidths=0.5
)
plt.title("Confusion Matrix — Token Level", fontsize=14)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Then we move to do correct prediction examples. This help to show examples where the model predicted correctly and understand what the model is confident and good at.

In [ ]:
print("=== Examples: Correct Predictions ===\n")
correct = results_df[results_df["match"] == True].reset_index(drop=True)

In [ ]:
for i, row in correct.head(5).iterrows():
    print(f"Sample {i+1}")
    print(f"  True      : {row['true_labels']}")
    print(f"  Predicted : {row['predicted_labels']}")
    print()

Then we do wrong prediction examples

In [ ]:
print("=== Examples: Wrong Predictions ===\n")
for i, row in mismatched.head(5).iterrows():
    true_tags = row["true_labels"].split()
    pred_tags = row["predicted_labels"].split()
    print(f"Sample {i+1}")
    print(f"  True      : {row['true_labels']}")
    print(f"  Predicted : {row['predicted_labels']}")
    # Highlight which positions differ
    diffs = [f"pos {j}" for j, (t, p) in enumerate(zip(true_tags, pred_tags)) if t != p]
    print(f"  Errors at : {', '.join(diffs)}")
    print()

##### Why Address AI Ethics in This Project?

Building a model is not just about accuracy.
We must also consider the broader impact of deploying NER systems in the real world. There three key dimensions of AI ethic.

## AI Ethics Analysis

### 1. Data Security
- Dataset used: WikiANN (Indonesian) — fully public, sourced from Wikipedia
- No sensitive personal data (ID numbers, addresses, phone numbers) is present
- All processing is done within Google Colab — no data is sent to third-party servers

### 2. User Privacy
- WikiANN contains publicly available encyclopedia text, not personal communications
- No individual's private information is used or exposed during training or inference
- If this model were deployed on real news data, anonymization of private individuals
  would be required before processing

### 3. AI Ethics & Bias
Potential biases identified in this model:

| Bias Type | Description | Example |
|-----------|-------------|---------|
| **Name bias** | Western names may be recognized better than local Indonesian names | "John" vs "Suprapto" |
| **Topic bias** | Model may perform better on political news vs cultural/local news | trained mostly on Wikipedia |
| **Entity imbalance** | `LOC` entities are more frequent than `ORG` — model may favor LOC predictions | seen in label distribution chart |

**Recommendation:** Evaluate the model separately on diverse topics
(politics, sports, local culture) to detect and document performance gaps.

In [ ]:
trainer.save_model("./ner-bert-indonesian-final")
tokenizer.save_pretrained("./ner-bert-indonesian-final")
print("Model saved to: ./ner-bert-indonesian-final")

# Verify by loading the model back
from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline

loaded_tokenizer = AutoTokenizer.from_pretrained("./ner-bert-indonesian-final")
loaded_model     = AutoModelForTokenClassification.from_pretrained("./ner-bert-indonesian-final")

# Test with a sample Indonesian sentence
ner_pipeline = pipeline("ner", model=loaded_model, tokenizer=loaded_tokenizer,
                         aggregation_strategy="simple")

test_sentence = "Jokowi meresmikan proyek Kereta Cepat Indonesia di Bandung."
output = ner_pipeline(test_sentence)

print(f"\nTest sentence: {test_sentence}\n")
print("=== NER Output ===")
for entity in output:
    print(f"  {entity['word']:<20} → {entity['entity_group']} (score: {entity['score']:.2f})")

## Actionable Recommendations

Based on the training results and error analysis, here are recommended next steps:

| # | Recommendation | Reason |
|---|---------------|--------|
| 1 | **Add more training data** | More diverse Indonesian text improves generalization |
| 2 | **Include Date entity** | Assignment requires Date but WikiANN does not have it — add custom annotated data |
| 3 | **Fine-tune on domain-specific data** | News articles from detik.com / kompas.com would improve real-world performance |
| 4 | **Address name bias** | Add more samples with local Indonesian names (Javanese, Batak, Papuan) |
| 5 | **Increase epochs carefully** | More epochs may improve F1 but risks overfitting — monitor validation loss |
| 6 | **Try IndoBERT** | `indobenchmark/indobert-base-p1` is pre-trained specifically on Indonesian text |

#### Project Complete

**Model:** `bert-base-multilingual-cased` fine-tuned on WikiANN Indonesian  
**Task:** Named Entity Recognition — PER, LOC, ORG  
**Saved to:** `./ner-bert-indonesian-final`